In [1]:
import os
import pandas as pd

In [2]:
with open('/u/rfechner/data/eic_gsm8k_deduplicated/test.parquet', 'rb') as file:
    dataset = pd.read_parquet(file)

In [3]:
dataset

,prompt,correct_answer,incorrect_answer,correct_solution,incorrect_solution,explanation,error_type,wrong_step,source_file,line_idx
0,John decides to start collecting art. He pays ...,67500.0,69750.0,"The first 3 pieces each cost 45000/3=$15,000\n...","The first 3 pieces each cost 45000/3=$15,000\n...",Step 3 adds information that is not mentioned ...,Hallucination,3,data/generated_cases_GSM8K/adding_irrelevant_i...,0
1,Irene earns $500 if she works for 40 hours a w...,700.0,750.0,"If Irene worked 50 hours last week, the total ...","If Irene worked 50 hours last week, the total ...",Step 3 adds information that is not mentioned ...,Hallucination,3,data/generated_cases_GSM8K/adding_irrelevant_i...,1
2,Hadley wore his cowboy boots everywhere. He wa...,6.0,7.0,"After he walked 2 miles to the grocery store, ...","After he walked 2 miles to the grocery store, ...",Step 1 adds information that is not mentioned ...,Hallucination,1,data/generated_cases_GSM8K/adding_irrelevant_i...,2
3,John decides to start collecting art. He pays...,67500.0,74250.0,"The first 3 pieces each cost 45000/3=$15,000\n...","The first 3 pieces each cost 45000/3=$15,000\n...",Step 4 adds information that is not mentioned ...,Hallucination,4,data/generated_cases_GSM8K/adding_irrelevant_i...,3
4,John's neighbor tells him to walk his dog for ...,160.0,180.0,"April has 30 days, so if he didn't walk the do...","April has 30 days, so if he didn't walk the do...",Step 3 adds information that is not mentioned ...,Hallucination,3,data/generated_cases_GSM8K/adding_irrelevant_i...,4
...,...,...,...,...,...,...,...,...,...,...
889,Minnie is making a playlist of songs for a par...,3.0,0.5,An hour is equal to 60 minutes.\nMinnie has 16...,An hour is equal to 50 minutes.\nMinnie has 16...,"In the transformed solution, an incorrect unit...",Unit Conversion Error,1,data/generated_cases_GSM8K/unit_conversion_err...,89
890,The American Academy of Pediatrics recommended...,75.0,55.0,Mrs. Merril wants to follow 2 hours x 60 minut...,Mrs. Merril wants to follow 2 hours x 50 minut...,"Here, step 1 makes incorrect unit conversion, ...",Unit Conversion Error,1,data/generated_cases_GSM8K/unit_conversion_err...,90
891,"To make a cherry pie, Veronica needs 3 pounds ...",2.0,2.4,There are 80 cherries in a pound and she needs...,There are 80 cherries in a pound and she needs...,"Here, step 4 contains an incorrect unit conver...",Unit Conversion Error,4,data/generated_cases_GSM8K/unit_conversion_err...,91
896,Jaydee can type 38 words in a minute. How many...,2.0,2.4,Jaydee can type his research paper in 4560/38 ...,Jaydee can type his research paper in 4560/38 ...,"Here, step 2 makes an incorrect unit conversio...",Unit Conversion Error,2,data/generated_cases_GSM8K/unit_conversion_err...,96


In [4]:
dataset.columns

Index(['prompt', 'correct_answer', 'incorrect_answer', 'correct_solution',
       'incorrect_solution', 'explanation', 'error_type', 'wrong_step',
       'source_file', 'line_idx'],
      dtype='object')

In [5]:
# 1) construct prompt, once with correct, once with incorrect response.

rows = []
for i, row in dataset.iterrows():
    correct, incorrect = \
        {
            'prompt' : [{'role' : 'user', 'content' : row['prompt']}, 
                        {'role' : 'assistant', 'content' : row['correct_solution']}],
            'behaviour_type' : 'anchor',
            'source_file' : row['source_file'],
            'index' : i
        }, \
        {
            'prompt' : [{'role' : 'user', 'content' : row['prompt']}, 
                        {'role' : 'assistant', 'content' : row['incorrect_solution']}],
            'behaviour_type' : row['error_type'],
            'source_file' : row['source_file'],
            'index' : i
        }
    rows.extend([correct, incorrect])

In [6]:
df = pd.DataFrame(rows)

In [7]:
with open('/u/rfechner/data/eic_gsm8k_generated/delta.jsonl', 'w') as file:
    df.to_json(file, lines=True, orient='records')